# ContourMark: IconShop inference-time watermark (Colab)

Runs the real IconShop model with and without the geometric inference-time watermark and evaluates detection.
Read `colab/README.md` in the repository for what each step should produce and what to do on failure.

**Before running:** Runtime → Change runtime type → GPU. Run the cells in order. Do not edit repository files.

## 1. Persistent storage
Results go to Google Drive so a disconnected session can resume. Set `USE_DRIVE = False` to keep them in the session only.

In [ ]:
USE_DRIVE = True
RUN_NAME = 'run1'   # use a new name whenever you change settings below
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = f'/content/drive/MyDrive/contourmark_runs/{RUN_NAME}'
else:
    OUT = f'/content/runs/{RUN_NAME}'
print('results directory:', OUT)

## 2. Clone and install
Colab already has a CUDA build of torch; it is not reinstalled.

In [ ]:
%cd /content
!test -d svg-contourmark || git clone -q https://github.com/milhud/svg-contourmark.git
%cd /content/svg-contourmark
!git rev-parse HEAD
!pip install -q -e '.[test,eval]' transformers safetensors huggingface_hub einops matplotlib
!apt-get -qq install -y librsvg2-bin potrace > /dev/null || echo 'apt install failed: retrace attacks will be reported as errors'
!git pull -q
!npm install --silent
!node --input-type=module -e "import('svgo').then(() => console.log('svgo ok'))"

## 3. Download the IconShop checkpoint (about 540 MB)

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download('m1357l/iconshop-svg-generator', local_dir='models/iconshop',
                  allow_patterns=['*.json', '*.py', '*.safetensors', '*.pt', 'deepsvg/**/*.py', 'model/**/*.py'])
print('model ready')

## 4. Check the installation
Expected: all tests pass, and CUDA is available. If not, stop and report.

In [ ]:
!OMP_NUM_THREADS=1 python -m pytest -q tests/test_geosample.py tests/test_review_regressions.py tests/test_hf_generate.py
import torch
assert torch.cuda.is_available(), 'No GPU: change the runtime type to GPU'
print(torch.cuda.get_device_name(0))

## 5. Smoke test
Expected: 16 lines; plain rows near 0; marked rows usually more negative (often only −1 to −4 at top-p 0.5).

In [ ]:
%cd /content/svg-contourmark/experiments
!rm -rf /content/smoke
!python iconshop_geosample.py --output /content/smoke --prompts star heart house car --samples-per-prompt 2 --batch 8
import json
for line in open('/content/smoke/samples.jsonl'):
    r = json.loads(line)
    print(r['marked'], r['prompt'], len(r['tokens']), 'truncated' if r['truncated'] else 'complete', round(r['log10_p'], 2))
%cd /content/svg-contourmark

## 6. Full run
30 prompts × `SEEDS` × (plain + marked) per top-p value. Re-run this cell after a disconnect: it resumes.
To change settings, change `RUN_NAME` in step 1 first (a run directory is locked to its configuration).

In [ ]:
import os
os.environ.update({'OUT': OUT, 'TOP_PS': '0.5 0.9', 'SEEDS': '4', 'BATCH': '16', 'WORKERS': '2', 'STAGE': 'all'})
!bash colab/run_single_gpu.sh

## 7. Results to report

In [ ]:
import glob, json
print(open(f'{OUT}/commit.txt').read(), open(f'{OUT}/environment.txt').read())
for path in sorted(glob.glob(f'{OUT}/*_attacks.summary.md')):
    print('=' * 100); print(path); print(open(path).read())
for path in sorted(glob.glob(f'{OUT}/*_null.json')):
    print(path, json.load(open(path))['levels'])

## 8. Download (only needed without Drive)

In [ ]:
if not USE_DRIVE:
    !cd /content && zip -qr contourmark_results.zip runs && ls -lh contourmark_results.zip
    from google.colab import files
    files.download('/content/contourmark_results.zip')

## 9. Run 2: sampler variants, more top-p values, second key, CLIP quality
Run this only after run 1 worked. It needs the cells in sections 1–4 to have been run in this session (and `svgo ok` printed).
About 2.5 hours on an A100 at `SEEDS=8`; set `SEEDS` to `'4'` for about half. Re-run the cell after a disconnect: it resumes.

In [ ]:
import os
OUT2 = OUT.rsplit('/', 1)[0] + '/run2'
os.environ.update({'OUT': OUT2, 'SEEDS': '8', 'BATCH': '16', 'WORKERS': '2', 'STAGE': 'all'})
%cd /content/svg-contourmark
!bash colab/run2.sh

In [ ]:
import glob, json
print(open(f'{OUT2}/commit.txt').read(), open(f'{OUT2}/environment.txt').read())
print(open(f'{OUT2}/RUN_SUMMARY.md').read())
for path in sorted(glob.glob(f'{OUT2}/*_null.json')):
    d = json.load(open(path)); print(path.split('/')[-1], d.get('levels', d.get('note')))

In [ ]:
# Download run 2 as a zip (also saved on Drive if mounted)
!cd "$(dirname {OUT2})" && zip -qr /content/contourmark_run2.zip run2 && ls -lh /content/contourmark_run2.zip
from google.colab import files
files.download('/content/contourmark_run2.zip')